# 00 - Configuration (schema v7)

Run this first. Every other notebook reads `cache/config.json`.

The app prices a trip three ways: real Ryanair round-trip fares, an
accommodation (Airbnb) estimate, and an on-the-ground spend driven by the user's
chosen lifestyle (dinners, drinks, coffees, self-catered days) at real local
prices.

## What's in the config

- Group size and trip-length defaults (frontend sliders)
- Build-time date window
- Origins (BRU + CRL) and home location
- Baggage options (a flight cost)
- Ground-transport multiplier for gems (airport-to-town transfer)
- Lifestyle slider defaults

## Pipeline order
1. `00_config` (here)
2. `01_destinations`
3. `02_flights` - Ryanair fare calendars
4. `03_costs` - on-the-ground lifestyle price basket
5. `03b_accommodation` - Airbnb nightly-rate anchors (Inside Airbnb)
6. `04_combined` - merge destinations + fares + costs + accommodation
7. `05_export_app` - write app_data.json

## 1. Schema version, group, dates

In [1]:
from datetime import date, datetime, timezone
from pathlib import Path
import json

SCHEMA_VERSION = 7   # v7: adds the accommodation (Airbnb) price layer

# Group defaults (frontend sliders)
GROUP_SIZE_DEFAULT       = 7
TRIP_LENGTH_DAYS_DEFAULT = 7

# Build-time date window
START_DATE = date.today()
END_DATE   = date(2026, 8, 31)

# Origins
ORIGINS   = ["BRU", "CRL"]
CURRENCY  = "EUR"
HOME_CITY = "Brussels"
HOME_LAT  = 50.8466
HOME_LON  = 4.3528

print(f"Schema v{SCHEMA_VERSION}")
print(f"Defaults: {GROUP_SIZE_DEFAULT} pax, {TRIP_LENGTH_DAYS_DEFAULT}d")
print(f"Range: {START_DATE} to {END_DATE}, origins {ORIGINS}")

Schema v7
Defaults: 7 pax, 7d
Range: 2026-06-07 to 2026-08-31, origins ['BRU', 'CRL']


## 2. Europe definition

In [2]:
EUROPE_COUNTRIES = {
    "AT","BE","BG","HR","CY","CZ","DK","EE","FI","FR","DE","GR","HU",
    "IE","IT","LV","LT","LU","MT","NL","PL","PT","RO","SK","SI","ES","SE",
    "CH","IS","LI","NO","GB",
    "AL","BA","ME","MK","RS","XK",
    "AD","MC","SM","VA",
    "UA","MD","BY",
}
print(f"Europe: {len(EUROPE_COUNTRIES)} countries")

Europe: 45 countries


## 3. Baggage

In [3]:
BAGGAGE_OPTIONS = {
    "small":         {"label": "Small cabin (free)", "per_direction_eur": 0.0},
    "priority_10kg": {"label": "10 kg priority",     "per_direction_eur": 25.0},
    "checked_20kg":  {"label": "20 kg checked",      "per_direction_eur": 42.0},
}
BAGGAGE_DEFAULT = "priority_10kg"

## 4. Ground transport (for gems)

`destinations_master.py` stores `nearest_airports = [(IATA, minutes, eur), ...]` per gem.
The `eur` is one-way per-person. Multiply by 2 for round trip.

In [4]:
GROUND_TRANSPORT_RT_MULTIPLIER = 2.0

## 5. Lifestyle defaults

Starting positions for the app's lifestyle sliders - "a typical vacation," per
person. The user adjusts these; each is priced at the destination's real local
rate (see notebook `03_costs`).

In [5]:
# Per person. A relaxed week-long trip ("Balanced" profile in the app).
LIFESTYLE_DEFAULTS = {
    "dinners_per_week":           5,
    "lunches_per_week":           4,
    "fastfood_per_week":          2,
    "drinks_per_week":            7,
    "club_nights_per_week":       1,
    "coffees_per_day":            1,
    "self_catered_days_per_week": 2,
}
print("Lifestyle defaults:", LIFESTYLE_DEFAULTS)

Lifestyle defaults: {'dinners_per_week': 5, 'lunches_per_week': 4, 'fastfood_per_week': 2, 'drinks_per_week': 7, 'club_nights_per_week': 1, 'coffees_per_day': 1, 'self_catered_days_per_week': 2}


## 6. Save config

In [6]:
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)
OUT = CACHE_DIR / "config.json"

config = {
    "schema_version":     SCHEMA_VERSION,
    "generated_at":       datetime.now(timezone.utc).isoformat(),

    "group_size_default":        GROUP_SIZE_DEFAULT,
    "trip_length_days_default":  TRIP_LENGTH_DAYS_DEFAULT,

    "start_date": START_DATE.isoformat(),
    "end_date":   END_DATE.isoformat(),
    "origins":    ORIGINS,
    "currency":   CURRENCY,
    "home_city":  HOME_CITY,
    "home_lat":   HOME_LAT,
    "home_lon":   HOME_LON,

    "europe_countries": sorted(EUROPE_COUNTRIES),

    "baggage_options": BAGGAGE_OPTIONS,
    "baggage_default": BAGGAGE_DEFAULT,

    "ground_transport_rt_multiplier": GROUND_TRANSPORT_RT_MULTIPLIER,

    "lifestyle_defaults": LIFESTYLE_DEFAULTS,
}

OUT.write_text(json.dumps(config, indent=2), encoding="utf-8")
print(f"Wrote {OUT} - {len(config)} keys")

Wrote cache\config.json - 16 keys
